<a href="https://colab.research.google.com/github/sundaybest3/txtanalysis/blob/function_position/position_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

# 1. GitHub raw CSV links
textbook_url = "https://raw.githubusercontent.com/sundaybest3/txtanalysis/function_position/High%20School%20Textbook%20Dialogue%20data_first%20analysis.xlsx.csv"

sbcsae_urls = [
    "https://raw.githubusercontent.com/sundaybest3/txtanalysis/refs/heads/function_position/cleaned_sbcase_47_1.csv",
    "https://raw.githubusercontent.com/sundaybest3/txtanalysis/refs/heads/function_position/cleaned_sbcase_49_1.csv",
    "https://raw.githubusercontent.com/sundaybest3/txtanalysis/refs/heads/function_position/cleaned_sbcase_50_1.csv",
    "https://raw.githubusercontent.com/sundaybest3/txtanalysis/refs/heads/function_position/cleaned_sbcase_51_1.csv",
    "https://raw.githubusercontent.com/sundaybest3/txtanalysis/refs/heads/function_position/cleaned_sbcase_56_1.csv",
]

# 2. Read textbook file
textbook_df = pd.read_csv(textbook_url, encoding='latin1')
textbook_df["corpus"] = "textbook"

# 3. Read and combine SBCSAE files
sbcsae_dfs = []
for url in sbcsae_urls:
    df = pd.read_csv(url)
    df["corpus"] = "sbcsae"
    sbcsae_dfs.append(df)

sbcsae_df = pd.concat(sbcsae_dfs, ignore_index=True)

# 4. Merge all data
df_all = pd.concat([textbook_df, sbcsae_df], ignore_index=True)

print(df_all.head())
print(df_all["corpus"].value_counts())

  Publisher Author       Grade Level  Lesson No.        Section Title  \
0       YBM   Park  Common English 1         1.0  Build Comprehension   
1       YBM   Park  Common English 1         1.0  Build Comprehension   
2       YBM   Park  Common English 1         1.0  Build Comprehension   
3       YBM   Park  Common English 1         1.0  Build Comprehension   
4       YBM   Park  Common English 1         1.0  Build Comprehension   

              Subsection Label    Conversation ID Speech type  Turn No.  ...  \
0  Listen and Understand    K1  JP_C1_L1_BC_LU_01    Dialogue       1.0  ...   
1  Listen and Understand    K1  JP_C1_L1_BC_LU_01    Dialogue       2.0  ...   
2  Listen and Understand    K1  JP_C1_L1_BC_LU_01    Dialogue       3.0  ...   
3  Listen and Understand    K1  JP_C1_L1_BC_LU_01    Dialogue       4.0  ...   
4  Listen and Understand    K1  JP_C1_L1_BC_LU_01    Dialogue       5.0  ...   

  Agreement    corpus  speaker script position1 function1 position2 function2  \

In [8]:
# 필요한 열만 추출
data = df_all[["corpus", "position1", "agreement"]].copy()

# 범주형으로 변환
data["corpus"] = data["corpus"].astype("category")
data["position1"] = data["position1"].astype("category")
data["agreement"] = data["agreement"].astype("category")

print(data.head())

     corpus position1 agreement
0  textbook       NaN       NaN
1  textbook       NaN       NaN
2  textbook       NaN       NaN
3  textbook       NaN       NaN
4  textbook       NaN       NaN


In [10]:
count_data = (
    data.groupby(["corpus", "position1", "agreement"])
    .size()
    .reset_index(name="count")
)

print(count_data.head(20))

    corpus position1 agreement  count
0   sbcsae        TF        I1      0
1   sbcsae        TF        I2      4
2   sbcsae        TF        T1      0
3   sbcsae        TF        T2      0
4   sbcsae        TF        T3      0
5   sbcsae        TF        T4      0
6   sbcsae        TI        I1      3
7   sbcsae        TI        I2     14
8   sbcsae        TI        T1     16
9   sbcsae        TI        T2      2
10  sbcsae        TI        T3      4
11  sbcsae        TI        T4     28
12  sbcsae        UI        I1      4
13  sbcsae        UI        I2      5
14  sbcsae        UI        T1     38
15  sbcsae        UI        T2      7
16  sbcsae        UI        T3     12
17  sbcsae        UI        T4     66
18  sbcsae        UM        I1      0
19  sbcsae        UM        I2      0


/tmp/ipykernel_12599/3035966905.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.groupby(["corpus", "position1", "agreement"])


In [11]:
import itertools

corpora = ["textbook", "sbcsae"]
positions = ["TI", "UI", "UM", "TF"]
functions = ["T1", "T2", "T3", "T4", "I1", "I2"]

all_combinations = pd.DataFrame(
    list(itertools.product(corpora, positions, functions)),
    columns=["corpus", "position1", "agreement"]
)

count_data = all_combinations.merge(
    count_data,
    on=["corpus", "position1", "agreement"],
    how="left"
)

count_data["count"] = count_data["count"].fillna(0).astype(int)

print(count_data)

      corpus position1 agreement  count
0   textbook        TI        T1      0
1   textbook        TI        T2      0
2   textbook        TI        T3      0
3   textbook        TI        T4      0
4   textbook        TI        I1      0
5   textbook        TI        I2      0
6   textbook        UI        T1      0
7   textbook        UI        T2      0
8   textbook        UI        T3      0
9   textbook        UI        T4      0
10  textbook        UI        I1      0
11  textbook        UI        I2      0
12  textbook        UM        T1      0
13  textbook        UM        T2      0
14  textbook        UM        T3      0
15  textbook        UM        T4      0
16  textbook        UM        I1      0
17  textbook        UM        I2      0
18  textbook        TF        T1      0
19  textbook        TF        T2      0
20  textbook        TF        T3      0
21  textbook        TF        T4      0
22  textbook        TF        I1      0
23  textbook        TF        I2      0


In [12]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 범주형 지정
count_data["corpus"] = count_data["corpus"].astype("category")
count_data["position1"] = count_data["position1"].astype("category")
count_data["agreement"] = count_data["agreement"].astype("category")

# 1. Main effects model
model_main = smf.glm(
    formula="count ~ corpus + position1 + agreement",
    data=count_data,
    family=sm.families.Poisson()
).fit()

# 2. Two-way interaction model
model_2way = smf.glm(
    formula="count ~ (corpus + position1 + agreement)**2",
    data=count_data,
    family=sm.families.Poisson()
).fit()

# 3. Three-way interaction model
model_3way = smf.glm(
    formula="count ~ corpus * position1 * agreement",
    data=count_data,
    family=sm.families.Poisson()
).fit()

print(model_main.summary())
print(model_2way.summary())
print(model_3way.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  count   No. Observations:                   48
Model:                            GLM   Df Residuals:                       38
Model Family:                 Poisson   Df Model:                            9
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -47.463
Date:                Mon, 06 Apr 2026   Deviance:                       38.103
Time:                        05:47:10   Pearson chi2:                     52.1
No. Iterations:                    25   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.9908      0

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)


In [13]:
from scipy.stats import chi2

def compare_models(smaller, larger, name1, name2):
    lr_stat = smaller.deviance - larger.deviance
    df_diff = smaller.df_resid - larger.df_resid
    p_value = chi2.sf(lr_stat, df_diff)

    print(f"{name1} vs {name2}")
    print(f"Likelihood ratio chi-square = {lr_stat:.3f}")
    print(f"df = {df_diff}")
    print(f"p = {p_value:.6f}")
    print("-" * 40)

compare_models(model_main, model_2way, "Main effects model", "Two-way interaction model")
compare_models(model_2way, model_3way, "Two-way interaction model", "Three-way interaction model")

Main effects model vs Two-way interaction model
Likelihood ratio chi-square = 38.103
df = 23
p = 0.024832
----------------------------------------
Two-way interaction model vs Three-way interaction model
Likelihood ratio chi-square = -0.000
df = 15
p = 1.000000
----------------------------------------


In [14]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.glm(
    formula="count ~ corpus * position1 * agreement",
    data=count_data,
    family=sm.families.Poisson()
).fit()

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  count   No. Observations:                   48
Model:                            GLM   Df Residuals:                        0
Model Family:                 Poisson   Df Model:                           47
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -28.412
Date:                Mon, 06 Apr 2026   Deviance:                   5.3206e-09
Time:                        05:52:35   Pearson chi2:                 2.66e-09
No. Iterations:                    24   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/statsmodels/regression/_tools.py:121: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generaliz

In [16]:
model = smf.glm(
    formula="count ~ corpus * position1 + corpus * agreement + position1 * agreement",
    data=count_data,
    family=sm.families.Poisson()
).fit()

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  count   No. Observations:                   48
Model:                            GLM   Df Residuals:                       15
Model Family:                 Poisson   Df Model:                           32
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -28.412
Date:                Mon, 06 Apr 2026   Deviance:                   3.4896e-09
Time:                        05:53:18   Pearson chi2:                 1.74e-09
No. Iterations:                    25   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
